# MiniTienda - Sistema de Registro y Análisis de Ventas
**Lógica de Programación - UIDE**

Sistema de consola que mantiene catálogo, registra ventas, calcula métricas y genera gráficos.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from datetime import datetime

# ==================== ESTRUCTURAS DE DATOS ====================
# TUPLAS: Catálogo inmutable de productos
catalogo = (
    ('P001', 'Laptop'),
    ('P002', 'Mouse'),
    ('P003', 'Teclado'),
    ('P004', 'Monitor'),
    ('P005', 'Headphones')
)

# DICCIONARIOS: Precios y stock
precios = {
    'P001': 800.00,
    'P002': 25.00,
    'P003': 75.00,
    'P004': 300.00,
    'P005': 120.00
}

stock = {
    'P001': 5,
    'P002': 50,
    'P003': 40,
    'P004': 8,
    'P005': 20
}

# LISTAS: Buffer de ventas
ventas_registro = []

print("✓ Estructuras inicializadas")
print(f"Catálogo (TUPLAS): {catalogo}")
print(f"Precios (DICCIONARIOS): {precios}")
print(f"Stock (DICCIONARIOS): {stock}")

In [ ]:
# ==================== FUNCIONES ====================

def obtener_nombre_producto(producto_id):
    """Busca nombre en la tupla catálogo"""
    try:
        for pid, nombre in catalogo:
            if pid == producto_id:
                return nombre
        return None
    except Exception as e:
        registrar_error(f"Error al obtener nombre: {e}")
        return None

def validar_producto(producto_id):
    """Valida si el producto existe"""
    for pid, _ in catalogo:
        if pid == producto_id:
            return True
    return False

def registrar_error(mensaje):
    """Escribe intentos fallidos en log.txt"""
    try:
        with open('log.txt', 'a') as f:
            timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            f.write(f"[{timestamp}] {mensaje}\n")
    except IOError:
        print("⚠ No se pudo escribir en log.txt")

def registrar_venta(producto_id, cantidad, precio_unitario):
    """Registra venta en lista y aplica descuento si >= 10 unidades"""
    # Reto C: Descuento si cantidad >= 10
    descuento = 0.05 if cantidad >= 10 else 0.0
    precio_final = precio_unitario * (1 - descuento)
    total = precio_final * cantidad
    
    venta = {
        'producto_id': producto_id,
        'nombre': obtener_nombre_producto(producto_id),
        'cantidad': cantidad,
        'precio_unitario': precio_unitario,
        'descuento_pct': descuento * 100,
        'total': total,
        'fecha': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    }
    ventas_registro.append(venta)
    return venta

def guardar_csv():
    """Guarda ventas en ventas.csv usando PANDAS"""
    try:
        df = pd.DataFrame(ventas_registro)
        df.to_csv('ventas.csv', index=False)
        print(f"✓ CSV guardado: {len(ventas_registro)} ventas")
        return True
    except Exception as e:
        registrar_error(f"Error al guardar CSV: {e}")
        print(f"✗ Error al guardar CSV: {e}")
        return False

def cargar_csv():
    """Carga ventas desde ventas.csv"""
    try:
        if os.path.exists('ventas.csv'):
            df = pd.read_csv('ventas.csv')
            print(f"✓ CSV cargado: {len(df)} registros")
            return df
        else:
            registrar_error("Intento de cargar CSV inexistente")
            print("✗ Archivo ventas.csv no existe")
            return None
    except FileNotFoundError:
        registrar_error("FileNotFoundError: ventas.csv no encontrado")
        print("✗ Archivo no encontrado")
        return None
    except Exception as e:
        registrar_error(f"Error al cargar CSV: {e}")
        print(f"✗ Error: {e}")
        return None

print("✓ Funciones definidas")

In [ ]:
# ==================== MENÚ PRINCIPAL ====================

def menu():
    """Menú interactivo con while y control de flujo"""
    while True:
        print("\n" + "="*50)
        print("         MiniTienda - Sistema de Ventas")
        print("="*50)
        print("1) Ver catálogo")
        print("2) Registrar venta")
        print("3) Ver resumen de ventas")
        print("4) Calcular métricas (NumPy)")
        print("5) Graficar ingresos por producto")
        print("6) Exportar gráfico a PNG")
        print("7) Agregar producto al catálogo")
        print("8) Cargar CSV")
        print("9) Salir")
        print("="*50)
        
        try:
            opcion = input("Seleccione opción (1-9): ").strip()
            
            # Control de flujo con if/elif/else
            if opcion == '1':
                mostrar_catalogo()
            elif opcion == '2':
                realizar_venta()
            elif opcion == '3':
                resumen_ventas()
            elif opcion == '4':
                calcular_metricas()
            elif opcion == '5':
                graficar_ingresos()
            elif opcion == '6':
                exportar_grafico()
            elif opcion == '7':
                agregar_producto()
            elif opcion == '8':
                cargar_csv()
            elif opcion == '9':
                print("\n✓ Guardando datos...")
                guardar_csv()
                print("¡Hasta luego!")
                break
            else:
                print("✗ Opción inválida. Intente de nuevo.")
        except KeyboardInterrupt:
            print("\n✓ Programa interrumpido. Guardando...")
            guardar_csv()
            break
        except Exception as e:
            registrar_error(f"Error en menú: {e}")
            print(f"✗ Error: {e}")

print("✓ Función menú definida")

In [ ]:
# ==================== OPCIONES DEL MENÚ ====================

def mostrar_catalogo():
    """Muestra catálogo (TUPLAS)"""
    print("\n" + "-"*60)
    print("CATÁLOGO DE PRODUCTOS")
    print("-"*60)
    print(f"{'ID':<8} {'Nombre':<20} {'Precio':<12} {'Stock':<10}")
    print("-"*60)
    
    # Iteramos sobre la TUPLA catálogo
    for producto_id, nombre in catalogo:
        precio = precios[producto_id]
        stk = stock[producto_id]
        print(f"{producto_id:<8} {nombre:<20} ${precio:<11.2f} {stk:<10}")
    print("-"*60)

def realizar_venta():
    """Registra una venta con validación completa (try/except)"""
    print("\n" + "-"*60)
    print("REGISTRAR VENTA")
    print("-"*60)
    
    try:
        producto_id = input("ID del producto: ").strip().upper()
        
        # Reto D: Validar que producto existe en catálogo
        if not validar_producto(producto_id):
            mensaje = f"Intento de venta con producto_id inválido: {producto_id}"
            registrar_error(mensaje)
            print(f"✗ {mensaje}")
            return
        
        cantidad = int(input("Cantidad: "))
        
        if cantidad <= 0:
            print("✗ La cantidad debe ser mayor a 0")
            return
        
        if cantidad > stock[producto_id]:
            print(f"✗ Stock insuficiente. Disponible: {stock[producto_id]}")
            registrar_error(f"Intento de venta con stock insuficiente: {producto_id}, solicitado: {cantidad}")
            return
        
        # Registrar venta
        venta = registrar_venta(producto_id, cantidad, precios[producto_id])
        
        # Actualizar stock
        stock[producto_id] -= cantidad
        
        # Mostrar confirmación
        print(f"\n✓ Venta registrada:")
        print(f"  Producto: {venta['nombre']}")
        print(f"  Cantidad: {cantidad} unidades")
        if venta['descuento_pct'] > 0:
            print(f"  Descuento: {venta['descuento_pct']}%")
        print(f"  Total: ${venta['total']:.2f}")
        print("-"*60)
        
    except ValueError:
        registrar_error("Error: Entrada inválida (no numérica)")
        print("✗ Error: Ingrese valores numéricos válidos")
    except KeyError:
        registrar_error(f"KeyError: producto_id no existe en diccionarios")
        print("✗ Error: Producto no encontrado")
    except Exception as e:
        registrar_error(f"Error en realizar_venta: {e}")
        print(f"✗ Error inesperado: {e}")

print("✓ Funciones de menú definidas")

In [ ]:
def resumen_ventas():
    """Muestra resumen usando PANDAS DataFrame"""
    if not ventas_registro:
        print("\n✗ No hay ventas registradas")
        return
    
    print("\n" + "-"*80)
    print("RESUMEN DE VENTAS")
    print("-"*80)
    
    # Crear DataFrame con PANDAS
    df = pd.DataFrame(ventas_registro)
    print(df.to_string(index=False))
    print("-"*80)
    
    # GroupBy para totales por producto
    print("\nTOTAL POR PRODUCTO (PANDAS GroupBy):")
    totales = df.groupby('nombre')['total'].sum().sort_values(ascending=False)
    for producto, total in totales.items():
        print(f"  {producto}: ${total:.2f}")
    print("-"*80)

def calcular_metricas():
    """Calcula métricas usando NUMPY"""
    if not ventas_registro:
        print("\n✗ No hay ventas para calcular")
        return
    
    print("\n" + "-"*60)
    print("MÉTRICAS CON NUMPY")
    print("-"*60)
    
    # Extraer totales en arreglo NumPy
    totales = np.array([v['total'] for v in ventas_registro])
    cantidades = np.array([v['cantidad'] for v in ventas_registro])
    
    try:
        # NumPy: mean, sum, std
        print(f"Total de ingresos (np.sum): ${np.sum(totales):.2f}")
        print(f"Promedio por venta (np.mean): ${np.mean(totales):.2f}")
        print(f"Desv. estándar (np.std): ${np.std(totales):.2f}")
        print(f"Venta mínima (np.min): ${np.min(totales):.2f}")
        print(f"Venta máxima (np.max): ${np.max(totales):.2f}")
        print(f"Total de unidades vendidas: {np.sum(cantidades):.0f}")
        
        # División segura
        if np.sum(cantidades) > 0:
            promedio_precio = np.sum(totales) / np.sum(cantidades)
            print(f"Precio promedio por unidad: ${promedio_precio:.2f}")
        
    except ZeroDivisionError:
        registrar_error("Error: División por cero en cálculo de métricas")
        print("✗ Error: No se puede dividir por cero")
    except Exception as e:
        registrar_error(f"Error en calcular_metricas: {e}")
        print(f"✗ Error: {e}")
    
    print("-"*60)

print("✓ Funciones de análisis definidas")

In [ ]:
def graficar_ingresos():
    """Genera gráfico de barras con MATPLOTLIB"""
    if not ventas_registro:
        print("\n✗ No hay ventas para graficar")
        return
    
    try:
        df = pd.DataFrame(ventas_registro)
        
        # Agrupar por producto
        ingresos = df.groupby('nombre')['total'].sum().sort_values(ascending=False)
        
        # Crear gráfico MATPLOTLIB
        plt.figure(figsize=(10, 6))
        colores = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A', '#98D8C8']
        bars = plt.bar(range(len(ingresos)), ingresos.values, color=colores[:len(ingresos)])
        
        plt.xlabel('Producto', fontsize=12, fontweight='bold')
        plt.ylabel('Ingresos ($)', fontsize=12, fontweight='bold')
        plt.title('Ingresos por Producto - MiniTienda', fontsize=14, fontweight='bold')
        plt.xticks(range(len(ingresos)), ingresos.index, rotation=45, ha='right')
        
        # Agregar valores en barras
        for i, (bar, valor) in enumerate(zip(bars, ingresos.values)):
            plt.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                     f'${valor:.2f}', ha='center', va='bottom', fontweight='bold')
        
        plt.tight_layout()
        plt.show()
        print("✓ Gráfico mostrado")
        
    except Exception as e:
        registrar_error(f"Error al graficar: {e}")
        print(f"✗ Error: {e}")

def exportar_grafico():
    """Reto B: Exporta gráfico a PNG"""
    if not ventas_registro:
        print("\n✗ No hay ventas para graficar")
        return
    
    try:
        df = pd.DataFrame(ventas_registro)
        ingresos = df.groupby('nombre')['total'].sum().sort_values(ascending=False)
        
        plt.figure(figsize=(10, 6))
        colores = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A', '#98D8C8']
        bars = plt.bar(range(len(ingresos)), ingresos.values, color=colores[:len(ingresos)])
        
        plt.xlabel('Producto', fontsize=12, fontweight='bold')
        plt.ylabel('Ingresos ($)', fontsize=12, fontweight='bold')
        plt.title('Ingresos por Producto - MiniTienda', fontsize=14, fontweight='bold')
        plt.xticks(range(len(ingresos)), ingresos.index, rotation=45, ha='right')
        
        for i, (bar, valor) in enumerate(zip(bars, ingresos.values)):
            plt.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                     f'${valor:.2f}', ha='center', va='bottom', fontweight='bold')
        
        plt.tight_layout()
        plt.savefig('ingresos.png', dpi=300, bbox_inches='tight')
        print("✓ Gráfico exportado a ingresos.png")
        plt.close()
        
    except Exception as e:
        registrar_error(f"Error al exportar gráfico: {e}")
        print(f"✗ Error: {e}")

print("✓ Funciones de gráficos definidas")

In [ ]:
# Reto A y otras extensiones

def agregar_producto():
    """Reto A: Agrega nuevo producto al catálogo"""
    global catalogo, precios, stock
    
    print("\n" + "-"*60)
    print("AGREGAR NUEVO PRODUCTO")
    print("-"*60)
    
    try:
        producto_id = input("ID del nuevo producto (ej: P006): ").strip().upper()
        
        # Validar que no exista
        if validar_producto(producto_id):
            print(f"✗ El producto {producto_id} ya existe")
            return
        
        nombre = input("Nombre del producto: ").strip()
        precio = float(input("Precio: $"))
        stk = int(input("Stock inicial: "))
        
        if precio <= 0 or stk < 0:
            print("✗ Precio debe ser positivo y stock no negativo")
            return
        
        # Convertir tupla a lista, agregar, reconvertir
        catalogo = tuple(list(catalogo) + [(producto_id, nombre)])
        precios[producto_id] = precio
        stock[producto_id] = stk
        
        print(f"\n✓ Producto {producto_id} agregado exitosamente")
        print(f"  Nombre: {nombre}")
        print(f"  Precio: ${precio:.2f}")
        print(f"  Stock: {stk}")
        
    except ValueError:
        registrar_error("Error: Valores inválidos al agregar producto")
        print("✗ Error: Ingrese valores válidos")
    except Exception as e:
        registrar_error(f"Error al agregar producto: {e}")
        print(f"✗ Error: {e}")
    
    print("-"*60)

print("✓ Función de extensión definida")

## Celdas de Prueba - Ejecución del Sistema

In [ ]:
# PRUEBA 1: Generar datos automáticos para cumplir requisito de 10 ventas
print("\n" + "="*60)
print("PRUEBA AUTOMÁTICA: Generando 12 ventas de ejemplo")
print("="*60)

# Simulación de ventas automáticas
ventas_automaticas = [
    ('P001', 1, 800.00),   # Venta normal
    ('P002', 15, 25.00),   # Con descuento (>= 10)
    ('P003', 5, 75.00),    # Venta normal
    ('P004', 2, 300.00),   # Venta normal
    ('P005', 12, 120.00),  # Con descuento
    ('P002', 8, 25.00),    # Venta normal
    ('P001', 1, 800.00),   # Venta normal
    ('P003', 10, 75.00),   # Con descuento
    ('P004', 3, 300.00),   # Venta normal
    ('P002', 20, 25.00),   # Con descuento
    ('P005', 5, 120.00),   # Venta normal
    ('P003', 7, 75.00),    # Venta normal
]

for producto_id, cantidad, precio in ventas_automaticas:
    try:
        if cantidad <= stock[producto_id]:
            venta = registrar_venta(producto_id, cantidad, precio)
            stock[producto_id] -= cantidad
            nombre = venta['nombre']
            print(f"✓ {nombre}: {cantidad} unid. → ${venta['total']:.2f}")
        else:
            print(f"✗ Stock insuficiente para {producto_id}")
    except Exception as e:
        print(f"✗ Error: {e}")

print(f"\n✓ {len(ventas_registro)} ventas registradas")

In [ ]:
# PRUEBA 2: Guardar CSV
print("\nGUARDANDO DATOS A CSV...")
guardar_csv()

In [ ]:
# PRUEBA 3: Ver resumen (DataFrame)
print("\n" + "="*80)
print("PRUEBA: Ver Resumen de Ventas con PANDAS")
print("="*80)
resumen_ventas()

In [ ]:
# PRUEBA 4: Calcular métricas con NumPy
print("\n" + "="*60)
print("PRUEBA: Métricas con NumPy")
print("="*60)
calcular_metricas()

In [ ]:
# PRUEBA 5: Graficar ingresos
print("\n" + "="*60)
print("PRUEBA: Gráfico de Ingresos por Producto")
print("="*60)
graficar_ingresos()

In [ ]:
# PRUEBA 6: Exportar gráfico a PNG
print("\n" + "="*60)
print("PRUEBA: Exportar gráfico a PNG")
print("="*60)
exportar_grafico()

In [ ]:
# PRUEBA 7: Intentos fallidos (for error handling)
print("\n" + "="*60)
print("PRUEBA: Validación de errores")
print("="*60)

# Intentar vender producto inexistente
print("\nIntentando vender producto inexistente...")
if not validar_producto('P999'):
    registrar_error("Intento fallido: Producto P999 no existe")
    print("✗ Producto P999 no existe en catálogo")

# Intentar cantidad inválida
print("\nIntentando cantidad negativa...")
try:
    cantidad = -5
    if cantidad <= 0:
        raise ValueError("La cantidad debe ser positiva")
except ValueError as e:
    registrar_error(f"Intento con cantidad inválida: {e}")
    print(f"✗ {e}")

print("\n✓ Log.txt ha sido actualizado con los errores")

In [ ]:
# Ver contenido del log.txt
print("\n" + "="*60)
print("CONTENIDO DE LOG.TXT")
print("="*60)
try:
    with open('log.txt', 'r') as f:
        contenido = f.read()
        if contenido:
            print(contenido)
        else:
            print("(Sin errores registrados)")
except FileNotFoundError:
    print("Log.txt aún no creado")

## Menú Interactivo (Ejecutar para interacción completa)

In [ ]:
# ACTIVAR MENÚ INTERACTIVO
# Descomenta la siguiente línea para ejecutar el menú completo:
# menu()

print("\n✓ Sistema MiniTienda listo.")
print("\nPara ejecutar el menú interactivo, descomenta: menu()")
print("\nO ejecuta opcionalmente:")
print("  - mostrar_catalogo()")
print("  - resumen_ventas()")
print("  - calcular_metricas()")
print("  - graficar_ingresos()")